# IMPORT

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
import numpy as np

# Remonte de deux niveaux (si ton notebook est dans /notebooks/exploration.ipynb)
# pour atteindre la racine du projet
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from config.config import *

# Preprocessing

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


**Les images sont redimensionnées et normalisées selon les statistiques ImageNet, afin d’être compatibles avec le modèle ResNet pré-entraîné**

In [4]:
# Chargement du modèle resnet50 pour création des embeddings.
resnet = models.resnet50(pretrained=True)
# "Supprimer" la couche de classification
resnet.eval()

for param in resnet.parameters():
    param.requires_grad = False

# On créé notre propre modèle 
# *list(resnet.children())[:-1] : récupère toutes les couches de resnet50 sauf la dernière de classification
# nn.Sequential créé un nouveau modèle sur la base des couches et poids récupérés
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])


c:\Users\Fabien\Desktop\OC\P10\BrainScanAI\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Fabien\Desktop\OC\P10\BrainScanAI\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
def extract_features(image_path, model, transform) -> list :
    img = Image.open(image_path).convert('RGB')
    image = transform(img).unsqueeze(0)
    with torch.no_grad():
        features = model(image)

    return features.squeeze().numpy().tolist()

In [6]:
images_cancer_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "avec_labels" / "cancer"
images_normal_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "avec_labels" / "normal"
images_sans_label_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "sans_label"

In [ ]:
features = []

# On parcourt les dossiers
for path in tqdm([images_cancer_dir, images_normal_dir, images_sans_label_dir], desc="Progression des dossiers"):
    # On cherche les fichiers .jpg (vos images sont en .jpg)
    for image_path in tqdm(list(path.glob('*.jpg')), desc=f"Traitement {path.name}", leave=False):
        try:
            data = {
                'name' : image_path.name,
                'features' : extract_features(image_path, feature_extractor, transform).tolist()
            }
            features.append(data)
        except Exception as e:
            print(f"Erreur sur {image_path.name}: {e}")

# Création du DataFrame
df_features = pd.DataFrame(features)

# CORRECTION : On utilise df_features (le DataFrame) et non features (la liste)
# .iloc[0] permet de voir la taille du premier embedding extrait
if not df_features.empty:
    print(f"Forme d'un embedding : {len(df_features['features'].iloc[0])}")

# Sauvegarde

df_features.to_parquet(BASE_DIR / "mri_dataset_brain_cancer_oc" / "features.parquet")

Progression des dossiers: 100%|██████████| 3/3 [03:28<00:00, 69.61s/it]


AttributeError: 'list' object has no attribute 'shape'

In [9]:
df_features.to_parquet(BASE_DIR / "mri_dataset_brain_cancer_oc" / "features.parquet")